# WeatherPulse — Pipeline Summary
A light notebook that prints key stats from the database so you can verify the pipeline worked during a demo.

In [ ]:
import os
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=os.getenv("POSTGRES_PORT", "5432"),
    dbname=os.getenv("POSTGRES_DB", "weatherpulse"),
    user=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

## 1. Row counts per table

In [ ]:
tables = ["dim_city", "raw_weather", "fact_weather", "weather_alert"]
for t in tables:
    count = pd.read_sql(f"SELECT count(*) AS n FROM {t}", conn).iloc[0, 0]
    print(f"{t}: {count} rows")

## 2. Alerts triggered

In [ ]:
df_alerts = pd.read_sql(
    """
    SELECT c.name AS city, a.alert_type, count(*) AS count
    FROM weather_alert a
    JOIN dim_city c ON a.city_id = c.city_id
    GROUP BY c.name, a.alert_type
    ORDER BY count DESC;
    """, conn)
print(f"Total alerts: {len(df_alerts)}")
df_alerts if not df_alerts.empty else "No alerts found in the database."

## 3. Last fetch time

In [ ]:
df_last = pd.read_sql(
    "SELECT max(fetched_at) AS last_fetch FROM raw_weather", conn)
print(f"Last fetch: {df_last.iloc[0, 0]}")